# 第 1 周末练习 —— 技术问答解释器

## 练习目标（理念）

为了展示你对 **OpenAI API** 以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一段代码或一个技术问题
- **输出**：逐行清晰、严谨的解释（Markdown）
- **额外要求**：对 OpenAI 一侧用**流式（streaming）**一边生成一边刷新显示

这是你在课程期间自己也能天天用的工具：遇到看不懂的代码，丢进来问模型。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `chat.completions.create(...)` |
| `messages`（system / user） | system 定「怎么解释」，user 放具体代码问题 |
| 流式输出 `stream=True` | 边收 delta 边 `update_display` |
| OpenAI 云端模型 | 常量 `MODEL_GPT`（此处为 `gpt-5-nano`） |
| Ollama 本地模型 | 常量 `MODEL_LLAMA`（`llama3.2`），走 OpenAI 兼容的 `/v1` 接口 |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：至少有 `OPENAI_API_KEY`；本地需已启动 Ollama 并拉取 `llama3.2`
3. 在「提问」单元格改写 `question`，再分别跑 GPT 流式格与 Llama 格，对比回答风格


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：Markdown 渲染、display 初次显示、update_display 流式刷新同一块区域
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI 客户端类：调用云端（以及兼容 Ollama 的）Chat Completions API
from openai import OpenAI


In [ ]:
# ========== 常量：模型名字集中写在一处，后面只改这里 ==========

# OpenAI 云端模型名：字符串必须和账号可用的 model id 一致（此处为 gpt-5-nano）
MODEL_GPT = 'gpt-5-nano'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2；名字要和本机已安装的一致
MODEL_LLAMA = 'llama3.2'


In [ ]:
# ========== 环境检查 + system prompt + messages 组装 ==========

# 加载 .env：override=True 表示用文件里的值覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 从环境变量读取 OpenAI 密钥（常见名字 OPENAI_API_KEY）
api_key = os.getenv('OPENAI_API_KEY')

# 粗检：有值、以 sk-proj- 开头、长度看起来合理 → 认为密钥已配置
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    # 给人看的成功提示（英文文案保留，避免改行为/对照原文）
    print("OpenAI API Key was correctly setup.")
else:
    # 密钥缺失或格式不对时的提示（保留原文）
    print("OpenAPI API Key is missing.")

# 创建 OpenAI 客户端：默认会从环境变量读取 OPENAI_API_KEY
openai = OpenAI()

# system prompt：发给模型的「角色 + 回答规则」；保留英文，改译会改变回答风格/行为
technical_explainer_system_prompt = """
You will be asked code technical questions.
You will be provided with snippets of code that can be
the body of a function or the body of class,
explain every line and what it does,
remembering the context from the previous lines.
Respond in markdown without html tags.
"""

# 把 system / user 两条消息拼成 Chat Completions 需要的 messages 列表
def messages_for(system_prompt, user_question):
    return [
        # system：定调——怎么解释代码
        {"role": "system", "content": system_prompt},
        # user：真正的问题（代码片段 + 说明）
        {"role": "user", "content": user_question}
    ]


In [ ]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# 用户问题（含待解释代码）写在三引号字符串里；发给模型的内容保持英文/代码原样
question = """
Please explain what this code does and why:
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = soup.find_all("a")
    links = [link.get("href") for link in links if link.get("href")]
    return links
"""


In [ ]:
# ========== OpenAI 流式问答：边生成边刷新 Markdown ==========

# 定义流式技术问答函数：入参是用户问题字符串
def stream_technical_answer(user_question):
    # 调用 Chat Completions；stream=True 表示服务端持续推送增量 delta
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        # messages：system 规则 + user 问题
        messages=messages_for(technical_explainer_system_prompt, user_question),
        stream=True
    )
    # response：累积已收到的全部文本，用于整段 Markdown 刷新
    response = ""
    # 先放一个空的 Markdown 显示位，拿到 display_id，后面只更新这一块（不会刷屏）
    display_handle = display(Markdown(""), display_id=True)
    # 遍历流式事件：每个 chunk 可能带一小段 content
    for chunk in stream:
        # delta.content 可能是 None（例如结束帧）；用 or '' 避免把 None 拼进去
        response += chunk.choices[0].delta.content or ''
        # 用同一 display_id 刷新为「目前已累积的完整 Markdown」
        update_display(Markdown(response), display_id=display_handle.display_id)

# 对上面的 question 发起一次流式解释
stream_technical_answer(question)


In [ ]:
# ========== 本地 Llama：走 Ollama 的 OpenAI 兼容接口（非流式） ==========

# Ollama 对外暴露的 OpenAI 兼容基址（/v1）；需本机已启动 ollama serve
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# 再建一个 OpenAI 客户端，但 base_url 指到本地；api_key 对 Ollama 通常任意非空即可
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
# 非流式一次拿完整回答：model 用 MODEL_LLAMA，messages 与 GPT 侧同一套组装逻辑
response = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages_for(technical_explainer_system_prompt, question))
# 从 choices[0].message.content 取出模型正文
technical_answer = response.choices[0].message.content

# 在笔记本里用 Markdown 漂亮展示完整回答
display(Markdown(technical_answer))
